# Helpers

In [5]:
from pathlib import Path
import re
import pickle
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt


# ============================================================
# 0. USER SETTINGS
# ============================================================

PICKLE_DIR = Path(r"G:\My Drive\uc3m PhD\05 Analysis\01 Main\01 Stata\01 Main\03 Estimation\03 CS-DiD\out_colab\est")
OUT_DIR = Path(r"G:\My Drive\uc3m PhD\05 Analysis\01 Main\01 Stata\01 Main\04 Matching quality\out\05 cohort level diagnostics")
#OUT_DIR.mkdir(exist_ok=True, parents=True)

VALID_EXTENSIONS = ["*- out.pkl"]
SKIP_PATTERNS = [
    "lambda1 with PCs",
]
# Optional manual overrides.
# Only use this if some filenames do not contain a clean "p-4 - 12" pattern.
#
# Interpretation:
# pre_t = number of estimated pre-treatment ATT periods
# post_t = number of estimated post-treatment ATT periods
#
# Example:
# pre_t=3, post_t=12 gives rel_times:
# -3, -2, -1, 0, 1, ..., 11
#
SAMPLE_WINDOW_CONFIG = {
    # "M&A, top-tech": {"pre_t": 3, "post_t": 12},
    # "off-deal, top-tech": {"pre_t": 3, "post_t": 12},
}


# ============================================================
# 1. LOADING HELPERS
# ============================================================

def load_pickle(path: Path):
    with open(path, "rb") as f:
        return pickle.load(f)


def get_attr_or_key(obj, name):
    """
    Works if obj is either:
    - an object with obj.att_gt / obj.se_gt
    - a dict with obj["att_gt"] / obj["se_gt"]
    """
    if hasattr(obj, name):
        return getattr(obj, name)
    if isinstance(obj, dict) and name in obj:
        return obj[name]
    raise AttributeError(f"Could not find `{name}` in object.")


# ============================================================
# 2. FILENAME PARSING
# ============================================================

def parse_lambda_from_filename(filename: str):
    """
    Parses lambda0, lambda1, lambda0.35, lambda0.3500, etc.
    """
    m = re.search(r"lambda\s*([0-9]*\.?[0-9]+)", filename, flags=re.IGNORECASE)
    if not m:
        return np.nan
    return float(m.group(1))


def lambda_label(lam):
    """
    Labels lambda as:
    - lambda 0
    - lambda 1
    - optimal lambda
    """
    if np.isnan(lam):
        return "lambda unknown"
    if np.isclose(lam, 0):
        return "lambda 0"
    if np.isclose(lam, 1):
        return "lambda 1"
    return "optimal lambda"


def sample_name_from_filename(filename: str):
    """
    Takes everything before ', lambda...' as the sample name.

    Example:
    'M&A, top-tech, 80, 4q, caliper0.0500, nemogy, lambda1, p-4 - 12 - out.pkl'

    becomes:
    'M&A, top-tech, 80, 4q, caliper0.0500, nemogy'
    """
    stem = Path(filename).stem
    parts = re.split(r",\s*lambda", stem, flags=re.IGNORECASE)
    return parts[0].strip()


def parse_window_from_filename(filename: str):
    """
    Parses strings like:
        p-4 - 12

    Correct DiD logic:
        The filename gives the raw window from -4 to 12.
        But ATT(g,t) starts at -3 because -4 is the reference/baseline period.

    Therefore:
        p-4 - 12 -> rel_times = -3, -2, -1, 0, ..., 12
        p-4 - 11 -> rel_times = -3, -2, -1, 0, ..., 11

    Total periods:
        p-4 - 12 -> 16 periods
        p-4 - 11 -> 15 periods
    """
    m = re.search(r"p\s*(-?\d+)\s*-\s*(-?\d+)", filename, flags=re.IGNORECASE)

    if not m:
        return None

    left = int(m.group(1))    # e.g. -4
    right = int(m.group(2))   # e.g. 12

    if left >= 0:
        raise ValueError(
            f"Expected left side to be negative, e.g. p-4. Got {left} in {filename}"
        )

    baseline = left

    # Main correction:
    rel_times = np.arange(left + 1, right + 1)

    pre_t = int(np.sum(rel_times < 0))
    post_t = int(np.sum(rel_times >= 0))

    return pre_t, post_t, rel_times, baseline

def get_window_config(filename: str):
    """
    Priority:
    1. SAMPLE_WINDOW_CONFIG manual rule
    2. Parse p-4 - 12 from filename using the correct DiD logic
    """

    for key, cfg in SAMPLE_WINDOW_CONFIG.items():
        if key.lower() in filename.lower():
            left = int(cfg["left"])     # example: -4
            right = int(cfg["right"])   # example: 12

            baseline = left
            rel_times = np.arange(left + 1, right + 1)

            pre_t = int(np.sum(rel_times < 0))
            post_t = int(np.sum(rel_times >= 0))

            return pre_t, post_t, rel_times, baseline

    parsed = parse_window_from_filename(filename)

    if parsed is None:
        raise ValueError(
            f"Could not infer event window from filename: {filename}. "
            "Either add this sample to SAMPLE_WINDOW_CONFIG or include something like 'p-4 - 12'."
        )

    return parsed


# ============================================================
# 3. CONVERT ONE PICKLE TO LONG DATAFRAME
# ============================================================
def pickle_to_long_df(path: Path):
    obj = load_pickle(path)

    att = np.asarray(get_attr_or_key(obj, "att_gt")).reshape(-1)
    se = np.asarray(get_attr_or_key(obj, "se_gt")).reshape(-1)

    if att.shape != se.shape:
        raise ValueError(
            f"{path.name}: att_gt and se_gt have different shapes. "
            f"att_gt={att.shape}, se_gt={se.shape}"
        )

    pre_t, post_t, rel_times, baseline = get_window_config(path.name)

    periods_per_group = len(rel_times)
    n_obs = len(att)

    # --------------------------------------------------------
    # Standard case: complete G x T object
    # --------------------------------------------------------
    if n_obs % periods_per_group == 0:
        n_groups = n_obs // periods_per_group

        group_index = np.repeat(np.arange(n_groups), periods_per_group)
        period_index = np.tile(np.arange(periods_per_group), n_groups)
        rel_time_long = np.tile(rel_times, n_groups)

    # --------------------------------------------------------
    # M&A special case:
    # last cohort is missing the final post-treatment estimate
    # --------------------------------------------------------
    elif "m&a" in path.name.lower() and (n_obs + 1) % periods_per_group == 0:
        n_groups = (n_obs + 1) // periods_per_group

        group_index_list = []
        period_index_list = []
        rel_time_list = []

        for g in range(n_groups):
            if g == n_groups - 1:
                # last cohort: missing final post-treatment cell
                this_rel_times = rel_times[:-1]
                this_period_index = np.arange(periods_per_group - 1)
            else:
                this_rel_times = rel_times
                this_period_index = np.arange(periods_per_group)

            group_index_list.append(np.repeat(g, len(this_rel_times)))
            period_index_list.append(this_period_index)
            rel_time_list.append(this_rel_times)

        group_index = np.concatenate(group_index_list)
        period_index = np.concatenate(period_index_list)
        rel_time_long = np.concatenate(rel_time_list)

        if len(rel_time_long) != n_obs:
            raise ValueError(
                f"{path.name}: constructed rel_time length {len(rel_time_long)} "
                f"does not match att length {n_obs}."
            )

        print(
            f"Note: {path.name} treated as M&A special case. "
            "Last cohort missing final post-treatment cell."
        )

    # --------------------------------------------------------
    # Otherwise, something else is wrong
    # --------------------------------------------------------
    else:
        raise ValueError(
            f"{path.name}: att_gt length {n_obs} is not divisible by "
            f"periods_per_group={periods_per_group}.\n"
            f"Parsed rel_times={rel_times.tolist()}.\n"
            f"Parsed baseline={baseline}, pre_t={pre_t}, post_t={post_t}.\n"
            "This is not the recognized M&A missing-last-post-period case."
        )

    lam = parse_lambda_from_filename(path.name)
    sample = sample_name_from_filename(path.name)

    df = pd.DataFrame({
        "file": path.name,
        "sample": sample,
        "lambda": lam,
        "lambda_label": lambda_label(lam),
        "group_index": group_index,
        "period_index": period_index,
        "rel_time": rel_time_long,
        "baseline_rel_time": baseline,
        "att": att,
        "se": se,
    })

    df["is_pre"] = df["rel_time"] < 0
    df["is_post"] = df["rel_time"] >= 0

    df["t_stat"] = df["att"] / df["se"]
    df["abs_att"] = df["att"].abs()
    df["att_sq"] = df["att"] ** 2
    df["t_sq"] = df["t_stat"] ** 2

    return df



# Load all estimation files (out objects returned from Python run of csdid)

In [6]:

# ============================================================
# 4. LOAD ALL PICKLES
# ============================================================

all_paths = []

for ext in VALID_EXTENSIONS:
    all_paths.extend(PICKLE_DIR.glob(ext))

SKIP_PATTERNS = [
    "lambda1 with PCs",
]

all_paths = [
    p for p in all_paths
    if not any(pattern.lower() in p.name.lower() for pattern in SKIP_PATTERNS)
]

if len(all_paths) == 0:
    raise FileNotFoundError(f"No pickle files found in {PICKLE_DIR.resolve()}")

all_dfs = []

for path in sorted(all_paths):
    try:
        temp = pickle_to_long_df(path)
        all_dfs.append(temp)
        print(f"Loaded: {path.name}")
    except Exception as e:
        print(f"Skipping {path.name}")
        print(f"Reason: {e}")
        print("-" * 80)

if len(all_dfs) == 0:
    raise RuntimeError("No pickle files were successfully loaded.")

df = pd.concat(all_dfs, ignore_index=True)

print("\nLoaded files:", df["file"].nunique())
print("Loaded samples:", df["sample"].nunique())

print(
    df[["sample", "lambda", "lambda_label", "file"]]
    .drop_duplicates()
    .sort_values(["sample", "lambda"])
    .head(20)
)


# ============================================================
# 5. KEEP PRE-TREATMENT ATT(g,t)'s
# ============================================================

pre_df = df[df["is_pre"]].copy()

pre_individual = pre_df[
    [
        "sample",
        "lambda",
        "lambda_label",
        "file",
        "group_index",
        "rel_time",
        "att",
        "se",
        "t_stat",
    ]
].copy()



Note: M&A, baseline, 4q, caliper0.0500, nemogy, lambda0, p-4 - 10 - out.pkl treated as M&A special case. Last cohort missing final post-treatment cell.
Loaded: M&A, baseline, 4q, caliper0.0500, nemogy, lambda0, p-4 - 10 - out.pkl
Note: M&A, baseline, 4q, caliper0.0500, nemogy, lambda0.6, p-4 - 10 - out.pkl treated as M&A special case. Last cohort missing final post-treatment cell.
Loaded: M&A, baseline, 4q, caliper0.0500, nemogy, lambda0.6, p-4 - 10 - out.pkl
Note: M&A, baseline, 4q, caliper0.0500, nemogy, lambda1, p-4 - 10 - out.pkl treated as M&A special case. Last cohort missing final post-treatment cell.
Loaded: M&A, baseline, 4q, caliper0.0500, nemogy, lambda1, p-4 - 10 - out.pkl
Note: M&A, baseline, 4q, caliper0.1000, nemogy, lambda0, p-4 - 10 - out.pkl treated as M&A special case. Last cohort missing final post-treatment cell.
Loaded: M&A, baseline, 4q, caliper0.1000, nemogy, lambda0, p-4 - 10 - out.pkl
Note: M&A, baseline, 4q, caliper0.1000, nemogy, lambda0.6, p-4 - 10 - out.pk

# Calculate metrics

In [7]:

# ============================================================
# 6. COHORT-LEVEL METRICS
# ============================================================

def cohort_metrics(x):
    att = x["att"].to_numpy()
    se = x["se"].to_numpy()
    rel = x["rel_time"].to_numpy()

    t = att / se

    # Magnitude of pre-treatment deviations in original units.
    rms = np.sqrt(np.mean(att ** 2))

    # Signed average pre-treatment deviation.
    mean_pre = np.mean(att)

    # Precision-scaled RMS: RMS of t-statistics.
    wrms_tstat = np.sqrt(np.mean(t ** 2))

    # Precision-weighted RMS in original units.
    weights = 1 / (se ** 2)
    wrms_units = np.sqrt(np.sum(weights * att ** 2) / np.sum(weights))

    # Slope across pre-treatment periods.
    # If only one pre-period exists, slope is undefined.
    if len(np.unique(rel)) >= 2:
        pre_slope = np.polyfit(rel, att, deg=1)[0]
    else:
        pre_slope = np.nan

    return pd.Series({
        "RMS": rms,
        "MeanPre": mean_pre,
        "WRMS_tstat": wrms_tstat,
        "WRMS_units": wrms_units,
        "PreSlope": pre_slope,
        "n_pre_periods": len(att),
    })


cohort_summary = (
    pre_df
    .groupby(["sample", "lambda", "lambda_label", "file", "group_index"], as_index=False)
    .apply(cohort_metrics)
    .reset_index(drop=True)
)


# ============================================================
# 7. SAMPLE x LAMBDA METRICS
# ============================================================

def sample_lambda_metrics(x):
    att = x["att"].to_numpy()
    se = x["se"].to_numpy()

    t = att / se
    weights = 1 / (se ** 2)

    return pd.Series({
        "Aggregate_RMS": np.sqrt(np.mean(att ** 2)),
        "Aggregate_MeanPre": np.mean(att),
        "Aggregate_WRMS_tstat": np.sqrt(np.mean(t ** 2)),
        "Aggregate_WRMS_units": np.sqrt(np.sum(weights * att ** 2) / np.sum(weights)),
        "Median_abs_att": np.median(np.abs(att)),
        "P75_abs_att": np.quantile(np.abs(att), 0.75),
        "P90_abs_att": np.quantile(np.abs(att), 0.90),
        "P95_abs_att": np.quantile(np.abs(att), 0.95),
        "Max_abs_att": np.max(np.abs(att)),
        "N_pre_estimates": len(att),
        "N_groups": x["group_index"].nunique(),
        "N_pre_periods": x["rel_time"].nunique(),
    })


sample_summary = (
    pre_df
    .groupby(["sample", "lambda", "lambda_label", "file"], as_index=False)
    .apply(sample_lambda_metrics)
    .reset_index(drop=True)
)

print("\nSample-level pretrend summary:")
print(sample_summary)




Sample-level pretrend summary:
                                               sample  lambda    lambda_label  \
0            M&A, baseline, 4q, caliper0.0500, nemogy     0.0        lambda 0   
1            M&A, baseline, 4q, caliper0.0500, nemogy     0.6  optimal lambda   
2            M&A, baseline, 4q, caliper0.0500, nemogy     1.0        lambda 1   
3            M&A, baseline, 4q, caliper0.1000, nemogy     0.0        lambda 0   
4            M&A, baseline, 4q, caliper0.1000, nemogy     0.6  optimal lambda   
..                                                ...     ...             ...   
67  Off deal, top-tech, 80, 8q, caliper0.0500, nemogy     0.7  optimal lambda   
68  Off deal, top-tech, 80, 8q, caliper0.0500, nemogy     1.0        lambda 1   
69  Off deal, top-tech, 80, 8q, caliper0.1000, nemogy     0.0        lambda 0   
70  Off deal, top-tech, 80, 8q, caliper0.1000, nemogy     0.7  optimal lambda   
71  Off deal, top-tech, 80, 8q, caliper0.1000, nemogy     1.0        lambda 1

# Save

In [ ]:

# ============================================================
# 8. SAVE TABLES
# ============================================================

pre_individual.to_csv(OUT_DIR / "individual_pre_att_gt.csv", index=False)
cohort_summary.to_csv(OUT_DIR / "cohort_level_pretrend_metrics.csv", index=False)
sample_summary.to_csv(OUT_DIR / "sample_lambda_pretrend_summary.csv", index=False)

print(f"\nSaved tables to: {OUT_DIR.resolve()}")
https://chatgpt.com/share/69fd2d58-a7c4-8391-a938-8d8c6da24120



Saved tables to: G:\My Drive\uc3m PhD\05 Analysis\01 Main\01 Stata\01 Main\04 Matching quality\out\05 cohort level diagnostics


# Plot

In [ ]:

# ============================================================
# 9. PLOTTING HELPERS
# ============================================================

LAMBDA_ORDER = ["lambda 0", "optimal lambda", "lambda 1"]


def ordered_labels(data):
    labels = list(data["lambda_label"].dropna().unique())
    ordered = [x for x in LAMBDA_ORDER if x in labels]

    # Include any unexpected labels at the end.
    extra = [x for x in labels if x not in ordered]
    return ordered + extra


def safe_filename(s):
    s = re.sub(r"[^\w\-_\. ]", "_", str(s))
    s = s.replace(" ", "_")
    return s


def boxplot_by_lambda(data, value_col, title, ylabel, outpath):
    labels = ordered_labels(data)

    plot_data = [
        data.loc[data["lambda_label"].eq(label), value_col].dropna().to_numpy()
        for label in labels
    ]

    fig, ax = plt.subplots(figsize=(8, 5))
    ax.boxplot(plot_data, labels=labels, showmeans=True)
    ax.axhline(0, linewidth=1)
    ax.set_title(title)
    ax.set_ylabel(ylabel)
    ax.set_xlabel("Matching design")
    fig.tight_layout()
    fig.savefig(outpath, dpi=300)
    plt.close(fig)


def histogram_by_lambda(data, value_col, title, xlabel, outpath, bins=25):
    labels = ordered_labels(data)

    fig, ax = plt.subplots(figsize=(8, 5))

    for label in labels:
        vals = data.loc[data["lambda_label"].eq(label), value_col].dropna().to_numpy()
        if len(vals) > 0:
            ax.hist(vals, bins=bins, alpha=0.35, label=label)

    ax.axvline(0, linewidth=1)
    ax.set_title(title)
    ax.set_xlabel(xlabel)
    ax.set_ylabel("Count")
    ax.legend()
    fig.tight_layout()
    fig.savefig(outpath, dpi=300)
    plt.close(fig)


def lineplot_sample_summary(sample_sum, value_col, title, ylabel, outpath):
    labels = ordered_labels(sample_sum)

    xlabels = []
    yvals = []

    for label in labels:
        vals = sample_sum.loc[sample_sum["lambda_label"].eq(label), value_col].dropna()
        if len(vals) > 0:
            xlabels.append(label)
            yvals.append(vals.iloc[0])

    fig, ax = plt.subplots(figsize=(8, 5))
    ax.plot(xlabels, yvals, marker="o")
    ax.set_title(title)
    ax.set_ylabel(ylabel)
    ax.set_xlabel("Matching design")
    fig.tight_layout()
    fig.savefig(outpath, dpi=300)
    plt.close(fig)


def scatter_pre_att_by_group(data, title, outpath):
    """
    Shows each pre-treatment ATT(g,t), grouped by cohort and colored by lambda through separate panels.
    """
    labels = ordered_labels(data)

    for label in labels:
        sub = data[data["lambda_label"].eq(label)].copy()

        fig, ax = plt.subplots(figsize=(9, 5))

        for g, gdf in sub.groupby("group_index"):
            ax.plot(
                gdf["rel_time"],
                gdf["att"],
                marker="o",
                linewidth=1,
                alpha=0.5
            )

        ax.axhline(0, linewidth=1)
        ax.set_title(f"{title}\n{label}")
        ax.set_xlabel("Relative time")
        ax.set_ylabel("Pre-treatment ATT(g,t)")
        fig.tight_layout()

        label_safe = safe_filename(label)
        fig.savefig(outpath.parent / f"{outpath.stem}_{label_safe}.png", dpi=300)
        plt.close(fig)


# ============================================================
# 10. MAKE PLOTS BY SAMPLE
# ============================================================

for sample, sdf in pre_df.groupby("sample"):
    sname = safe_filename(sample)

    cdf = cohort_summary[cohort_summary["sample"].eq(sample)].copy()
    ssdf = sample_summary[sample_summary["sample"].eq(sample)].copy()

    # --------------------------------------------------------
    # A. Individual pre-treatment ATT(g,t)'s
    # --------------------------------------------------------

    boxplot_by_lambda(
        data=sdf,
        value_col="att",
        title=f"Individual pre-treatment ATT(g,t) by lambda\n{sample}",
        ylabel="Pre-treatment ATT(g,t)",
        outpath=OUT_DIR / f"{sname}_box_individual_pre_ATT.png"
    )

    histogram_by_lambda(
        data=sdf,
        value_col="att",
        title=f"Distribution of individual pre-treatment ATT(g,t)\n{sample}",
        xlabel="Pre-treatment ATT(g,t)",
        outpath=OUT_DIR / f"{sname}_hist_individual_pre_ATT.png"
    )

    scatter_pre_att_by_group(
        data=sdf,
        title=f"Cohort-specific pre-treatment ATT(g,t)\n{sample}",
        outpath=OUT_DIR / f"{sname}_cohort_paths_pre_ATT.png"
    )

    # --------------------------------------------------------
    # B. Cohort-level RMS
    # --------------------------------------------------------

    boxplot_by_lambda(
        data=cdf,
        value_col="RMS",
        title=f"Cohort-level RMS pre-treatment discrepancy\n{sample}",
        ylabel="RMS of pre-treatment ATT(g,t)",
        outpath=OUT_DIR / f"{sname}_box_cohort_RMS.png"
    )

    histogram_by_lambda(
        data=cdf,
        value_col="RMS",
        title=f"Distribution of cohort-level RMS\n{sample}",
        xlabel="Cohort-level RMS",
        outpath=OUT_DIR / f"{sname}_hist_cohort_RMS.png"
    )

    # --------------------------------------------------------
    # C. Cohort-level MeanPre
    # --------------------------------------------------------

    boxplot_by_lambda(
        data=cdf,
        value_col="MeanPre",
        title=f"Cohort-level signed mean pre-treatment discrepancy\n{sample}",
        ylabel="Mean pre-treatment ATT(g,t)",
        outpath=OUT_DIR / f"{sname}_box_cohort_MeanPre.png"
    )

    histogram_by_lambda(
        data=cdf,
        value_col="MeanPre",
        title=f"Distribution of signed cohort pre-trends\n{sample}",
        xlabel="Mean pre-treatment ATT(g,t)",
        outpath=OUT_DIR / f"{sname}_hist_cohort_MeanPre.png"
    )

    # --------------------------------------------------------
    # D. Cohort-level WRMS using t-statistics
    # --------------------------------------------------------

    boxplot_by_lambda(
        data=cdf,
        value_col="WRMS_tstat",
        title=f"Cohort-level precision-scaled WRMS\n{sample}",
        ylabel="RMS of pre-treatment t-statistics",
        outpath=OUT_DIR / f"{sname}_box_cohort_WRMS_tstat.png"
    )

    histogram_by_lambda(
        data=cdf,
        value_col="WRMS_tstat",
        title=f"Distribution of cohort-level precision-scaled WRMS\n{sample}",
        xlabel="RMS of pre-treatment t-statistics",
        outpath=OUT_DIR / f"{sname}_hist_cohort_WRMS_tstat.png"
    )

    # --------------------------------------------------------
    # E. Cohort-level pre-treatment slope
    # --------------------------------------------------------

    boxplot_by_lambda(
        data=cdf,
        value_col="PreSlope",
        title=f"Cohort-level pre-treatment slope\n{sample}",
        ylabel="Slope of pre-treatment ATT(g,t)",
        outpath=OUT_DIR / f"{sname}_box_cohort_PreSlope.png"
    )

    histogram_by_lambda(
        data=cdf,
        value_col="PreSlope",
        title=f"Distribution of cohort-level pre-treatment slopes\n{sample}",
        xlabel="Slope of pre-treatment ATT(g,t)",
        outpath=OUT_DIR / f"{sname}_hist_cohort_PreSlope.png"
    )

    # --------------------------------------------------------
    # F. Aggregate discrepancy by lambda
    # --------------------------------------------------------

    lineplot_sample_summary(
        sample_sum=ssdf,
        value_col="Aggregate_RMS",
        title=f"Aggregate RMS pre-treatment discrepancy by lambda\n{sample}",
        ylabel="Aggregate RMS",
        outpath=OUT_DIR / f"{sname}_line_aggregate_RMS.png"
    )

    lineplot_sample_summary(
        sample_sum=ssdf,
        value_col="Aggregate_WRMS_tstat",
        title=f"Aggregate precision-scaled WRMS by lambda\n{sample}",
        ylabel="Aggregate WRMS, t-stat scale",
        outpath=OUT_DIR / f"{sname}_line_aggregate_WRMS_tstat.png"
    )

    lineplot_sample_summary(
        sample_sum=ssdf,
        value_col="Aggregate_MeanPre",
        title=f"Aggregate signed mean pre-treatment discrepancy by lambda\n{sample}",
        ylabel="Aggregate MeanPre",
        outpath=OUT_DIR / f"{sname}_line_aggregate_MeanPre.png"
    )


print(f"\nDone. All outputs saved to: {OUT_DIR.resolve()}")